In [1]:
import h5py
import mne
import numpy as np
import pyedflib
import os
from pathlib import Path
import ddd
import re
from scipy.signal import decimate
import gc





Can't open C:\Users\Sindel\Documents\epiweek\d.file\Easrec-service25_250519-1152.d


TypeError: 'NoneType' object is not subscriptable

In [2]:

# Set of standard known labels — expand this if needed
known_labels = {
    'Fp1', 'Fp2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz',
    'EKG', 'CAL', 'SDT', 'EVT', 'A1', 'A2', 'O1', 'O2', 'P3', 'P4', 'C3', 'C4'
}

def split_channel_names_glued(raw, known_set):
    labels = []
    temp = ''
    i = 0
    while i < len(raw):
        temp += raw[i]
        # Try matching as longest label
        for j in range(1, 5):  # Try 1 to 4 chars
            candidate = raw[i:i + j]
            if candidate in known_set:
                labels.append(candidate)
                i += j
                temp = ''
                break
        else:
            i += 1  # No match; move forward
    return labels

In [4]:
folder_path = Path(r"C:\Users\Sindel\Documents\epiweek\d.file")
file_names = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
Save_folder = Path(r"C:\Users\Sindel\Documents\epiweek\vsechny")
print("Files in folder:", file_names)
print(Path(folder_path/file_names[1][:-3]))

extension = '.d'
files = list(folder_path.glob(f"*{extension}"))
print([f.name for f in files])
print(files)
file_names = [f.name for f in files]
print(f"Here are all files type d. {file_names}")
EDFs_p = Save_folder/ "EDFs"
EDFs_p.mkdir(parents=True, exist_ok=True)

Files in folder: ['Easrec-service25_250519-1152.d', 'Easrec-service25_250519-1517.d', 'Easrec-service25_250519-2023.d', 'Easrec-service25_250519-2102.d', 'Easrec-service25_250520-0013.d', 'Easrec-service25_250520-0108.d', 'Easrec-service25_250520-0319.d', 'Easrec-service25_250520-0443.d', 'Easrec-service25_250520-1058.d', 'Easrec-service25_250520-1133.d', 'Easrec-service25_250520-1136.d', 'Easrec-service25_250520-1158.d', 'Easrec-service25_250520-1555.d', 'Easrec-service25_250520-2037.d', 'Easrec-service25_250520-2103.d', 'Easrec-service25_250520-2124.d', 'Easrec-service25_250520-2234.d', 'Easrec-service25_250521-0036.d', 'Easrec-service25_250521-0344.d', 'Easrec-service25_250521-0539.d', 'Easrec-service25_250521-0611.d', 'Easrec-service25_250521-1010.d', 'Easrec-service25_250521-1642.d', 'Easrec-service25_250521-2300.d', 'Easrec-service25_250521-2334.d', 'Easrec-service25_250522-0035.d', 'Easrec-service25_250522-0222.d', 'Easrec-service25_250522-0340.d', 'Easrec-service25_250522-0524.

In [5]:
for name in file_names:

    d_path = Path(folder_path/name)
    h = ddd.getDRheader(d_path)
    raw = h['xheader'].get('channel_names', '')
    nchan = h['sheader']['nchan']
    channel_names = split_channel_names_glued(raw, known_labels)
    print(f"Parsed {len(channel_names)} channel names: {channel_names}")
    fs = h['sheader']['fsamp']
    lenOfFil = h['sheader']['nsamp']
    data = ddd.getDRdata(h, ch=list(range(1, h['sheader']['nchan'] + 1)),s1=0,s2=lenOfFil)

    # Create a simple Raw object
    
    # Downsample factor (e.g., 10 → from 1000 Hz to 100 Hz)
    factor = 10

    # Downsample each channel
    data_ds = decimate(data, factor, axis=1, ftype='fir')
    sfreq = fs/factor



    if data.shape[0] != len(channel_names):
        missing = data.shape[0] - len(channel_names)
        if missing > 0:
            # Add missing channel names
            new_channels = [f"X{i+1}" for i in range(missing)]
            channel_names.extend(new_channels)
        else:
            raise ValueError("More channel names than data rows!")

            
    info = mne.create_info(ch_names=channel_names, sfreq=sfreq, ch_types='eeg')
    #raw = mne.io.RawArray(data[1::3], info)
    raw = mne.io.RawArray(data_ds, info)   # --   HERE you have to use [1::2] bcs you want to pick filtered one - There are saved also raw and filtered data -- You take just every second signal  

    data = raw.get_data()  # shape: (n_channels, n_times)
    clean_data = data.copy()

    # Parameters
    threshold_factor = 7  # You can adjust this
    method = 'std'  # 'std' or 'mad'

    # Detect and remove outliers per channel
    for ch in range(clean_data.shape[0]):
        signal = clean_data[ch]
        
        if method == 'std':
            mean = np.mean(signal)
            std = np.std(signal)
            threshold = threshold_factor * std
        elif method == 'mad':
            median = np.median(signal)
            mad = np.median(np.abs(signal - median))
            threshold = threshold_factor * mad / 0.6745  # robust estimate of std
        else:
            raise ValueError("Unknown method")

        # Find outliers
        if method == 'std':
            outliers = np.abs(signal - mean) > threshold
        else:
            outliers = np.abs(signal - median) > threshold

        # Replace outliers with NaNs
        clean_data[ch, outliers] = np.nan

    # Optionally interpolate over NaNs (here with linear interpolation)
    for ch in range(clean_data.shape[0]):
        sig = clean_data[ch]
        nans = np.isnan(sig)
        if np.any(nans):
            clean_data[ch, nans] = np.interp(np.flatnonzero(nans),
                                            np.flatnonzero(~nans),
                                            sig[~nans])

    # Set cleaned data back
    raw._data = clean_data
    # Downsample to 2000 Hz
    #raw.resample(2000)

    # Analyze data
    play = raw._data
    print(type(play))
    print(play.shape)
    max_values = play.max(axis=1)
    min_values = play.min(axis=1)
    mean_values = play.mean(axis=1)
    std_values = play.std(axis=1)
    lenght_d = play.shape[1]
    print(lenght_d)
    sec = lenght_d/sfreq
    min = sec/60

    min = np.floor(min)

    # Print summary of each channel
    for i, ch_name in enumerate(raw.ch_names):
        print(f"Channel {ch_name}:")
        print(f"  Max: {max_values[i]:.2f}")
        print(f"  Min: {min_values[i]:.2f}")
        print(f"  Mean: {mean_values[i]:.2f}")
        print(f"  Std: {std_values[i]:.2f}")

    # Find the time of max and min values
    sfreq = raw.info['sfreq']
    times = np.arange(play.shape[1]) / sfreq
    max_times = times[np.argmax(play, axis=1)]
    min_times = times[np.argmin(play, axis=1)]

    # Print times of extrema
    for i, ch_name in enumerate(raw.ch_names):
        print(f"Channel {ch_name}:")
        print(f"  Max at {max_times[i]:.2f} s")
        print(f"  Min at {min_times[i]:.2f} s")

    # Scale data if necessary (e.g., from volts to microvolts)
    #scaling_factor = 1e3
    #scaled_data = play / scaling_factor

    # Save to EDF using pyedflib
    name = name[:-2]
    
    edf_file = Path(EDFs_p/name)
    edf_file = str(edf_file)
    print(edf_file)
    n_channels = play.shape[0]
    signal_headers = []

    # Create signal headers for each channel
    for ch_name in raw.info['ch_names']:
        
        signal_headers.append({
            'label': ch_name,
            'dimension': 'uV',
            'sample_frequency': sfreq,
            'physical_min': np.min(play),
            'physical_max': np.max(play),
            'digital_min': -32768,
            'digital_max': 32767,
            'transducer': '',
            'prefilter': ''
        })



    # for seky in range(0,int(np.floor(sec/duration_of_Edf-1))):
    #     print(play[:,int(seky*sfreq*duration_of_Edf):int((seky*sfreq*duration_of_Edf)+(1*sfreq*duration_of_Edf))].shape)
        

    # Write the EDF file
    with pyedflib.EdfWriter(edf_file+"W_2500sf.edf", n_channels, file_type=pyedflib.FILETYPE_EDFPLUS) as edf:
        edf.setSignalHeaders(signal_headers)
        edf.writeSamples(play)
        edf.close()

    print(f"CREATED::::\033[95m{edf_file}\033[0m####################")
    gc.collect()

Parsed 21 channel names: ['Fp1', 'Fp2', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz', 'EKG', 'CAL', 'SDT', 'EVT']
Creating RawArray with float64 data, n_channels=23, n_times=3162001
    Range : 0 ... 3162000 =      0.000 ...  1264.800 secs
Ready.
<class 'numpy.ndarray'>
(23, 3162001)
3162001
Channel Fp1:
  Max: 1637.04
  Min: -1636.90
  Mean: -0.22
  Std: 231.99
Channel Fp2:
  Max: 2617.47
  Min: -2613.81
  Mean: 0.23
  Std: 372.70
Channel C3:
  Max: 900.63
  Min: -898.63
  Mean: -0.18
  Std: 127.80
Channel C4:
  Max: 1781.16
  Min: -1780.78
  Mean: 0.03
  Std: 247.37
Channel P3:
  Max: 832.49
  Min: -832.79
  Mean: -0.44
  Std: 116.18
Channel P4:
  Max: 962.01
  Min: -961.98
  Mean: -0.13
  Std: 135.67
Channel O1:
  Max: 6674.21
  Min: -6671.52
  Mean: -2.25
  Std: 697.21
Channel O2:
  Max: 3445.37
  Min: -3446.28
  Mean: -2.42
  Std: 397.24
Channel F7:
  Max: 4602.44
  Min: -4600.51
  Mean: 0.92
  Std: 485.47
Channel F8:
  Max: 6126.47
  Mi

c:\Users\Sindel\anaconda3\envs\mne\Lib\site-packages\pyedflib\edfwriter.py:133: UserWarning: Physical minimum for channel 0 (Fp1) is -8866.01581671491, which has 17 chars, however, EDF+ can only save 8 chars, will be truncated to -8866.01, some loss of precision is to be expected
  warnings.warn('Physical minimum for channel {} ({}) is {}, which has {} chars, '
c:\Users\Sindel\anaconda3\envs\mne\Lib\site-packages\pyedflib\edfwriter.py:140: UserWarning: Physical maximum for channel 0 (Fp1) is 65655.57680904194, which has 17 chars, however, EDF+ can only save 8 chars, will be truncated to 65655.57, some loss of precision is to be expected.
  warnings.warn('Physical maximum for channel {} ({}) is {}, which has {} chars, '
c:\Users\Sindel\anaconda3\envs\mne\Lib\site-packages\pyedflib\edfwriter.py:133: UserWarning: Physical minimum for channel 1 (Fp2) is -8866.01581671491, which has 17 chars, however, EDF+ can only save 8 chars, will be truncated to -8866.01, some loss of precision is to be

CREATED::::C:\Users\Sindel\Documents\epiweek\vsechny\EDFs\Easrec-service25_250519-1152####################
Parsed 21 channel names: ['Fp1', 'Fp2', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz', 'EKG', 'CAL', 'SDT', 'EVT']
Creating RawArray with float64 data, n_channels=23, n_times=3217019
    Range : 0 ... 3217018 =      0.000 ...  1286.807 secs
Ready.
<class 'numpy.ndarray'>
(23, 3217019)
3217019
Channel Fp1:
  Max: 2234.10
  Min: -2035.10
  Mean: -0.14
  Std: 319.22
Channel Fp2:
  Max: 2752.41
  Min: -2692.39
  Mean: -0.02
  Std: 394.59
Channel C3:
  Max: 1075.51
  Min: -1062.16
  Mean: -0.08
  Std: 154.60
Channel C4:
  Max: 1579.67
  Min: -1581.72
  Mean: 0.09
  Std: 226.18
Channel P3:
  Max: 526.95
  Min: -527.09
  Mean: -0.13
  Std: 74.71
Channel P4:
  Max: 1898.86
  Min: -1898.42
  Mean: 1.25
  Std: 176.57
Channel O1:
  Max: 1981.80
  Min: -1981.14
  Mean: -0.43
  Std: 254.78
Channel O2:
  Max: 2046.58
  Min: -2046.25
  Mean: -1.77
  Std

c:\Users\Sindel\anaconda3\envs\mne\Lib\site-packages\pyedflib\edfwriter.py:133: UserWarning: Physical minimum for channel 0 (Fp1) is -5146.218110328343, which has 18 chars, however, EDF+ can only save 8 chars, will be truncated to -5146.21, some loss of precision is to be expected
  warnings.warn('Physical minimum for channel {} ({}) is {}, which has {} chars, '
c:\Users\Sindel\anaconda3\envs\mne\Lib\site-packages\pyedflib\edfwriter.py:133: UserWarning: Physical minimum for channel 1 (Fp2) is -5146.218110328343, which has 18 chars, however, EDF+ can only save 8 chars, will be truncated to -5146.21, some loss of precision is to be expected
  warnings.warn('Physical minimum for channel {} ({}) is {}, which has {} chars, '
c:\Users\Sindel\anaconda3\envs\mne\Lib\site-packages\pyedflib\edfwriter.py:133: UserWarning: Physical minimum for channel 2 (C3) is -5146.218110328343, which has 18 chars, however, EDF+ can only save 8 chars, will be truncated to -5146.21, some loss of precision is to b

CREATED::::C:\Users\Sindel\Documents\epiweek\vsechny\EDFs\Easrec-service25_250519-1517####################
Parsed 21 channel names: ['Fp1', 'Fp2', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz', 'EKG', 'CAL', 'SDT', 'EVT']
Creating RawArray with float64 data, n_channels=23, n_times=5697681
    Range : 0 ... 5697680 =      0.000 ...  2279.072 secs
Ready.
<class 'numpy.ndarray'>
(23, 5697681)
5697681
Channel Fp1:
  Max: 789.85
  Min: -790.10
  Mean: -0.72
  Std: 93.47
Channel Fp2:
  Max: 1339.27
  Min: -1339.45
  Mean: -1.40
  Std: 160.99
Channel C3:
  Max: 669.34
  Min: -669.22
  Mean: -1.21
  Std: 51.63
Channel C4:
  Max: 1225.79
  Min: -1225.79
  Mean: -0.79
  Std: 126.94
Channel P3:
  Max: 839.05
  Min: -838.89
  Mean: 0.08
  Std: 118.85
Channel P4:
  Max: 267.69
  Min: -267.57
  Mean: -0.17
  Std: 34.25
Channel O1:
  Max: 2342.69
  Min: -2342.79
  Mean: -3.71
  Std: 268.63
Channel O2:
  Max: 931.99
  Min: -931.92
  Mean: -1.21
  Std: 112.19


...

c:\Users\Sindel\anaconda3\envs\mne\Lib\site-packages\pyedflib\edfwriter.py:133: UserWarning: Physical minimum for channel 0 (Fp1) is -5559.440535381265, which has 18 chars, however, EDF+ can only save 8 chars, will be truncated to -5559.44, some loss of precision is to be expected
  warnings.warn('Physical minimum for channel {} ({}) is {}, which has {} chars, '
c:\Users\Sindel\anaconda3\envs\mne\Lib\site-packages\pyedflib\edfwriter.py:133: UserWarning: Physical minimum for channel 1 (Fp2) is -5559.440535381265, which has 18 chars, however, EDF+ can only save 8 chars, will be truncated to -5559.44, some loss of precision is to be expected
  warnings.warn('Physical minimum for channel {} ({}) is {}, which has {} chars, '
c:\Users\Sindel\anaconda3\envs\mne\Lib\site-packages\pyedflib\edfwriter.py:133: UserWarning: Physical minimum for channel 2 (C3) is -5559.440535381265, which has 18 chars, however, EDF+ can only save 8 chars, will be truncated to -5559.44, some loss of precision is to b

CREATED::::C:\Users\Sindel\Documents\epiweek\vsechny\EDFs\Easrec-service25_250523-1212####################
Parsed 21 channel names: ['Fp1', 'Fp2', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz', 'EKG', 'CAL', 'SDT', 'EVT']
Creating RawArray with float64 data, n_channels=23, n_times=4148592
    Range : 0 ... 4148591 =      0.000 ...  1659.436 secs
Ready.
<class 'numpy.ndarray'>
(23, 4148592)
4148592
Channel Fp1:
  Max: 1656.36
  Min: -1655.29
  Mean: 0.12
  Std: 236.04
Channel Fp2:
  Max: 2331.29
  Min: -2330.66
  Mean: 0.19
  Std: 331.49
Channel C3:
  Max: 1157.60
  Min: -1157.35
  Mean: -0.16
  Std: 122.16
Channel C4:
  Max: 1464.11
  Min: -1463.18
  Mean: 0.15
  Std: 207.30
Channel P3:
  Max: 615.71
  Min: -615.68
  Mean: -0.19
  Std: 80.34
Channel P4:
  Max: 520.18
  Min: -520.26
  Mean: 0.22
  Std: 70.53
Channel O1:
  Max: 831.83
  Min: -832.60
  Mean: 0.06
  Std: 118.55
Channel O2:
  Max: 833.64
  Min: -833.72
  Mean: -0.42
  Std: 110.75
C

c:\Users\Sindel\anaconda3\envs\mne\Lib\site-packages\pyedflib\edfwriter.py:133: UserWarning: Physical minimum for channel 0 (Fp1) is -28186.614759837044, which has 19 chars, however, EDF+ can only save 8 chars, will be truncated to -28186.6, some loss of precision is to be expected
  warnings.warn('Physical minimum for channel {} ({}) is {}, which has {} chars, '
c:\Users\Sindel\anaconda3\envs\mne\Lib\site-packages\pyedflib\edfwriter.py:133: UserWarning: Physical minimum for channel 1 (Fp2) is -28186.614759837044, which has 19 chars, however, EDF+ can only save 8 chars, will be truncated to -28186.6, some loss of precision is to be expected
  warnings.warn('Physical minimum for channel {} ({}) is {}, which has {} chars, '
c:\Users\Sindel\anaconda3\envs\mne\Lib\site-packages\pyedflib\edfwriter.py:133: UserWarning: Physical minimum for channel 2 (C3) is -28186.614759837044, which has 19 chars, however, EDF+ can only save 8 chars, will be truncated to -28186.6, some loss of precision is t

CREATED::::C:\Users\Sindel\Documents\epiweek\vsechny\EDFs\Easrec-service25_250523-1619####################
Parsed 0 channel names: []


ValueError: sfreq must be positive